# Reranker experiments — completing the paper's evidence — TPU edition

Four outstanding items sharing one expensive setup (corpus embedding + retrieval), so the index is
built **once**.

| | Item | Why |
|---|---|---|
| **A** | Fix `bge-reranker-base` | It scored *below* the no-rerank baseline — the signature of a 2D-score-array bug |
| **B** | Arabic-specific rerankers | `GATE-Reranker-V1`, `Namaa-ARA-Reranker-V1` |
| **C** | Rerank depth ablation | top-10/20/50 |
| **D** | **Asymmetry test** | paired difference-of-differences bootstrap for "reranking helps Darija more" |

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**`primary_depth` was not guaranteed to be among `depths`.** Sections A, B and D all index
`results[(method, field, primary_depth)]`. Set `primary_depth` to a value not in `depths` and
section D's `except KeyError: continue` skips *every* method and prints an empty asymmetry table —
which reads as "no effect" rather than as a configuration error. `primary_depth` is now asserted
into `depths` up front, and D reports any method it had to skip instead of swallowing it.

**The 2D-score guard was already correct** and is kept as-is — that was the point of item A.

### What makes the TPU real

1. **Explicit XLA device**, with the backend actually obtained printed — so a silent CPU fallback
   is never mistaken for a TPU run.
2. **Fixed input shapes.** XLA recompiles per tensor shape; every batch is padded to exactly
   `(batch_size, max_length)`, so each model compiles once instead of once per shape.
3. **Batched across queries** rather than one `.predict()` per query — far better utilisation.

Encoding and reranking are hand-rolled on `AutoModel` / `AutoModelForSequenceClassification` so
device and padding are under our control. The encoder was checked against `sentence-transformers`
and matches to within 3e-8, so retrieval numbers stay comparable to the GPU notebooks.

Falls back to CUDA then CPU automatically and says which it got.

### Install

In [ ]:
import importlib.util, subprocess, sys

# Pin torch_xla to the ALREADY-INSTALLED torch so pip does not pull a different
# torch and force a runtime restart mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
                        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
                       capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
else:
    print("torch_xla already available")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "sentencepiece"], check=False)
print("deps ready")

### Device — and which one we actually got

In [ ]:
import torch

BACKEND, device, xm = "cpu", torch.device("cpu"), None
try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"

def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()

print("=" * 80)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
print("=" * 80)

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,
    "depths": [10, 20, 50],          # C: rerank depth ablation
    "primary_depth": 20,             # depth used for A, B and D
    "rerankers": [
        "BAAI/bge-reranker-v2-m3",              # current best, for comparison
        "BAAI/bge-reranker-base",               # A: rerun with the scoring guard
        "NAMAA-Space/GATE-Reranker-V1",         # B: Arabic-specific, claims dialect coverage
        "NAMAA-Space/Namaa-ARA-Reranker-V1",    # B: second Arabic-specific option
    ],
    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
    "max_length": 512,
    "batch_size": 32,
    "checkpoint": "reranker_experiments_checkpoint.pkl",
}

# A, B and D all index results[(method, field, primary_depth)]. If primary_depth
# were not among depths, D's `except KeyError: continue` would silently skip every
# method and print an empty asymmetry table that reads as "no effect".
assert CONFIG["primary_depth"] in CONFIG["depths"], \
    f"primary_depth {CONFIG['primary_depth']} must be one of depths {CONFIG['depths']}"
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
print(f"Corpus {len(corpus)} | evaluating all {len(qa)} items")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Retrieve once at the DEEPEST depth; shallower depths are prefixes

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real); last batch padded up to bs to keep XLA shapes static."""
    for i in range(0, len(items), bs):
        ch = list(items[i:i + bs]); n = len(ch)
        if n < bs:
            ch += [ch[-1]] * (bs - n)
        yield ch, n

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(model, tk, texts, bs=32, max_len=512, label=""):
    """e5-style mean pooling + L2 normalisation. Verified to match
    sentence-transformers to within 3e-8 on this corpus."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(texts, bs):
        enc = tk(ch, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        v = mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        o.append(v.float().cpu().numpy()[:n]); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(o, 0).astype("float32")

@torch.no_grad()
def score_pairs(model, tk, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score per (query, passage) pair, static shapes for XLA."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(pairs, bs):
        enc = tk([a for a, _ in ch], [b for _, b in ch], padding="max_length",
                 truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Same guard as the GPU notebook: some rerankers emit a 2D per-class array.
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        o.extend(s[:n].tolist()); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)} ({time.time()-t0:.0f}s)", flush=True)
    return np.asarray(o)

print("XLA helpers ready")

In [ ]:
MAXD = max(CONFIG["depths"] + [CONFIG["primary_depth"]])
print(f"Building index on {BACKEND.upper()} and retrieving top-{MAXD}...")

enc_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
enc_model = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(enc_model, enc_tok, [f"passage: {t}" for t in corpus_texts], label="corpus")
q_emb = {f: encode_texts(enc_model, enc_tok, [f"query: {q[f]}" for q in qa], label=f)
         for f in ["msa_query", "darija_query"]}

candidates = {}
for field in ["msa_query", "darija_query"]:
    d = {}
    for i, q in enumerate(qa):
        s = (CONFIG["alpha"] * minmax(corpus_emb @ q_emb[field][i])
             + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q[field])))
        d[q["id"]] = [corpus_ids[j] for j in np.argsort(-s)[:MAXD]]
    candidates[field] = d
    print(f"  {field} done")

del enc_model, corpus_emb, q_emb
gc.collect()

### Evaluation helper + baselines

In [ ]:
def evaluate_order(ordered_by_qid):
    o = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            o[f"R@{k}"].append(1.0 if (pos and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in o.items()}, "MRR": np.array(rr)}

CKPT = out(CONFIG["checkpoint"])
results = {}
if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        results.update(pickle.load(f))
    print(f"Resumed {len(results)} cached entries.")

for depth in CONFIG["depths"]:
    for field in ["msa_query", "darija_query"]:
        key = ("no_rerank", field, depth)
        if key not in results:
            results[key] = evaluate_order({q["id"]: candidates[field][q["id"]][:depth] for q in qa})

print("\nBaseline (no reranking):")
for depth in CONFIG["depths"]:
    m = results[("no_rerank", "darija_query", depth)]
    print(f"  top-{depth:<3} Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}")

### A + B + C: every reranker at every depth, with the scoring guard

In [ ]:
def run_reranker(model_name):
    short = model_name.split("/")[-1]
    if all((short, f, d) in results for d in CONFIG["depths"] for f in ["msa_query", "darija_query"]):
        print(f"\n=== {short} === (cached, skipping)"); return
    print(f"\n=== {short} ===  on {BACKEND.upper()}")
    try:
        tk = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        ce = AutoModelForSequenceClassification.from_pretrained(
            model_name, trust_remote_code=True).to(device=device, dtype=torch.float32).eval()
    except Exception as e:
        print(f"  SKIPPED (load): {type(e).__name__}: {str(e)[:160]}"); return
    try:
        for field in ["msa_query", "darija_query"]:
            qids = [q["id"] for q in qa]
            flat = [(q[field], corpus_map[c]) for q in qa for c in candidates[field][q["id"]]]
            sc = score_pairs(ce, tk, flat, CONFIG["batch_size"], CONFIG["max_length"], field)
            # Score once at MAXD, then slice: reranking top-d is the first d
            # candidates of the same list reordered.
            for depth in CONFIG["depths"]:
                reordered = {}
                for i, qid in enumerate(qids):
                    cands = candidates[field][qid][:depth]
                    s = sc[i * MAXD:i * MAXD + depth]
                    reordered[qid] = [cands[j] for j in np.argsort(-s)]
                results[(short, field, depth)] = evaluate_order(reordered)
            print(f"  {field} done (all depths)")
    except Exception as e:
        print(f"  SKIPPED (inference): {type(e).__name__}: {str(e)[:160]}")
    finally:
        del ce
        gc.collect()
    with open(CKPT, "wb") as f:
        pickle.dump(results, f)

for name in CONFIG["rerankers"]:
    run_reranker(name)

### Results table

In [ ]:
rows = [{"method": m, "query": f, "depth": d, **{k: float(v.mean()) for k, v in mm.items()}}
        for (m, f, d), mm in results.items()]
df = pd.DataFrame(rows)
df.to_csv(out("reranker_experiments_full.csv"), index=False)

D = CONFIG["primary_depth"]
print("=" * 90); print(f"RESULTS AT TOP-{D}"); print("=" * 90)
print(df[df.depth == D].pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
      .to_string(float_format=lambda x: f"{x:.3f}"))

### A: did the bug fix change bge-reranker-base?

In [ ]:
print("\n" + "=" * 90); print("A. BUG FIX CHECK - bge-reranker-base"); print("=" * 90)
prev = {"msa_query": 0.520, "darija_query": 0.365}   # from the buggy run
sub = df[(df.method == "bge-reranker-base") & (df.depth == D)]
if len(sub):
    for _, r in sub.iterrows():
        base = df[(df.method == "no_rerank") & (df["query"] == r["query"]) & (df.depth == D)]["R@1"].iloc[0]
        print(f"  {r['query']:<14} before fix {prev[r['query']]:.3f} | after fix {r['R@1']:.3f} | "
              f"no-rerank baseline {base:.3f}")
    print("\n  At or above baseline -> the 2D-score bug explained it.")
    print("  Still below -> the model is genuinely unsuited; report it rather than dropping it.")
else:
    print("  Model did not run.")

### C: depth ablation

In [ ]:
print("\n" + "=" * 90); print("C. RERANK DEPTH ABLATION (Darija R@1)"); print("=" * 90)
print(df[df["query"] == "darija_query"].pivot(index="depth", columns="method", values="R@1")
      .to_string(float_format=lambda x: f"{x:.3f}"))
print("""
Deeper reranking raises the ceiling but costs proportionally more cross-encoder
passes. Flat or falling numbers with depth mean the extra candidates add noise.""")

### D: the asymmetry test (the paper's novel claim)

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def boot_ci(d):
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    m = d[idx].mean(axis=1)
    return d.mean(), *np.percentile(m, [2.5, 97.5])

print("\n" + "=" * 90)
print("D. ASYMMETRY TEST - does reranking help Darija MORE than MSA?")
print("=" * 90)
print("Two separate CIs on the two gains do NOT establish that the gains differ.")
print("This is a paired difference-of-differences bootstrap, which does.\n")

asym, skipped = [], []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    try:
        dar = results[(method, "darija_query", D)]["R@1"] - results[("no_rerank", "darija_query", D)]["R@1"]
        msa = results[(method, "msa_query", D)]["R@1"] - results[("no_rerank", "msa_query", D)]["R@1"]
    except KeyError:
        skipped.append(method)     # reported below, not silently swallowed
        continue
    dd, lo, hi = boot_ci(dar - msa)
    asym.append({"method": method, "darija_gain": dar.mean(), "msa_gain": msa.mean(),
                 "difference": dd, "lo": lo, "hi": hi,
                 "verdict": ("helps Darija more" if lo > 0 else
                             "helps MSA more" if hi < 0 else "no significant asymmetry")})

if skipped:
    print(f"  NOTE: no results at top-{D} for {', '.join(skipped)} - excluded from this test.\n")
asymdf = pd.DataFrame(asym)
print(asymdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}") if len(asymdf)
      else "  No method had results at this depth - nothing to test.")
asymdf.to_csv(out("asymmetry_test.csv"), index=False)
print("""
'helps Darija more' with a CI excluding zero is what the paper's novel claim
requires. 'no significant asymmetry' means the claim must be softened to a
descriptive observation rather than a demonstrated effect.""")

### Dialect gap by method

In [ ]:
print("\n" + "=" * 90); print(f"DIALECT GAP AT TOP-{D} (MSA - Darija, R@1)"); print("=" * 90)
gaps = []
for method in df.method.unique():
    try:
        d = results[(method, "msa_query", D)]["R@1"] - results[(method, "darija_query", D)]["R@1"]
    except KeyError:
        continue
    g, lo, hi = boot_ci(d)
    gaps.append({"method": method, "gap": g, "lo": lo, "hi": hi, "significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gaps).sort_values("gap")
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv(out("gap_by_reranker.csv"), index=False)

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["reranker_experiments_full.csv", "asymmetry_test.csv", "gap_by_reranker.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["reranker_experiments_full.csv", "asymmetry_test.csv", "gap_by_reranker.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")